# Lab 3 — Model Comparison: Formula 1 Point Scoring (Jolpica API)

**Framing:** Binary Classification. "Will this driver finish in the Top 10 (score points)?"
**Metric:** Macro F1-score.
**Reasoning:** The team needs to know if a mid-field car configuration has a realistic chance of reaching the points. A binary classifier focuses directly on the business outcome (scoring points = financial reward) rather than the precise ranking.


## 1. Setup & Imports
**Justification:** We need standard data manipulation libraries (pandas) and sklearn for validation and modeling. We fix the random seed to 414 for reproducibility as requested in the rubric.


In [5]:
import requests
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

RANDOM_SEED = 414
np.random.seed(RANDOM_SEED)


## 2. Data Ingestion (Jolpica API)
**Justification:** We use the Jolpica API (a community continuation of Ergast) to fetch real F1 results data. To prevent excessive API calls and timeout issues, we will fetch data from a restricted temporal window (e.g., 2021-2023).


In [6]:
import time

def fetch_f1_data(start_year=2021, end_year=2023):
    results_list = []
    # Añadimos un User-Agent porque a veces las APIs bloquean las consultas por defecto de Python
    headers = {
        'User-Agent': 'IIT414W-Student-Project/1.0'
    }
    
    for year in range(start_year, end_year + 1):
        # Cambiamos HTTP a HTTPS, que suele ser la causa de que los requests se queden colgados o den timeout
        url = f"https://api.jolpi.ca/ergast/f1/{year}/results.json?limit=1000"
        
        # Sistema de reintentos (3 intentos por año)
        for attempt in range(3):
            try:
                print(f"Descargando año {year} (Intento {attempt + 1}/3)...")
                # Agregamos timeout de 15 segundos para que no se quede colgado eternamente
                response = requests.get(url, headers=headers, timeout=15)
                
                if response.status_code == 200:
                    data = response.json()
                    races = data.get('MRData', {}).get('RaceTable', {}).get('Races', [])
                    for race in races:
                        for res in race.get('Results', []):
                            results_list.append({
                                'season': int(race['season']),
                                'round': int(race['round']),
                                'driver': res['Driver']['driverId'],
                                'constructor': res['Constructor']['constructorId'],
                                'grid': int(res['grid']),
                                'position': int(res.get('position', 20)),
                                'points': float(res['points'])
                            })
                    break # Éxito, rompe el ciclo de reintentos y pasa al siguiente año
                else:
                    print(f"Error en la API: Código {response.status_code}")
                    time.sleep(2) # Espera 2 segundos antes de reintentar
            except requests.exceptions.RequestException as e:
                print(f"Error de conexión/Timeout: {e}")
                time.sleep(3) # Espera 3 segundos si la conexión falló
                
    return pd.DataFrame(results_list)

df = fetch_f1_data(2021, 2023)
display(df.head())

Descargando año 2021 (Intento 1/3)...
Descargando año 2022 (Intento 1/3)...
Descargando año 2023 (Intento 1/3)...


,season,round,driver,constructor,grid,position,points
0,2021,1,hamilton,mercedes,2,1,25.0
1,2021,1,max_verstappen,red_bull,1,2,18.0
2,2021,1,bottas,mercedes,3,3,16.0
3,2021,1,norris,mclaren,7,4,12.0
4,2021,1,perez,red_bull,0,5,10.0


**Analysis (Data Ingestion)**
The data collected via the API matches real-world historical results, capturing the starting position (`grid`), and finishing outcomes for ~3 years. This raw layout sets up the foundation needed to generate our operational features.

## 3. Feature Engineering & Target Definition
**Justification:** Our chosen framing is binary: did the driver score points or not (finish <= 10). The target `scored_points` is 1 if points > 0 else 0. For features, we will use the `grid` position (qualifying performance) and encode the constructor.


In [7]:
# Target Definition
df['scored_points'] = (df['points'] > 0).astype(int)

# Feature Engineering: One-hot encode constructor ID to capture car performance
df = pd.get_dummies(df, columns=['constructor'], drop_first=True)

# Define our features (X) and target (y)
feature_cols = ['grid'] + [col for col in df.columns if col.startswith('constructor_')]
X = df[['season', 'round'] + feature_cols].copy()
y = df['scored_points']

print("Target distribution:")
print(y.value_counts(normalize=True))


Target distribution:
scored_points
1    0.5
0    0.5
Name: proportion, dtype: float64


**Analysis (Features & Target)**
As shown in the target distribution, exactly half the cars (10 out of 20) score points, which theoretically implies a 50/50 split. However, technical failures or varying grid sizes sometimes slightly shift this ratio. The relative class balance confirms our choice of `Macro F1` is highly appropriate, treating both "Points" and "No Points" as equally important classes.

## 4. Temporal Validation Split
**Justification:** As required by the rubric ("Temporal validation only. No random splits"), we will use a walk-forward / chronological split. We will train on the 2021-2022 seasons and test on the 2023 season.


In [8]:
# Temporal Split: Train on 2021-2022, Test on 2023
train_mask = X['season'] < 2023
test_mask = X['season'] == 2023

X_train = X[train_mask][feature_cols]
y_train = y[train_mask]

X_test = X[test_mask][feature_cols]
y_test = y[test_mask]

print(f"Train size: {len(X_train)} rows")
print(f"Test size: {len(X_test)} rows")


Train size: 200 rows
Test size: 100 rows


**Analysis (Temporal Split)**
By using a forward temporal split along the 2023 boundary, our testing environment precisely mirrors real-world racing deployments. We prevent "look-ahead bias" or "time leakage" which is a massive failure mode when dealing with random splits in sports analytics.

## 5. Model 1: Baseline - Majority Class
**Justification:** A naive baseline. Predicts the most frequent class in the training set (which is usually 0, indicating not scoring points) for all rows. This proves our models learn something beyond base rates.


In [9]:
class MajorityBaseline:
    def fit(self, X, y):
        self.majority_class_ = y.mode()[0]
    def predict(self, X):
        return np.full(len(X), self.majority_class_)

baseline_1 = MajorityBaseline()
baseline_1.fit(X_train, y_train)

y_pred_b1_train = baseline_1.predict(X_train)
y_pred_b1_test = baseline_1.predict(X_test)

b1_train_mf1 = f1_score(y_train, y_pred_b1_train, average='macro')
b1_test_mf1 = f1_score(y_test, y_pred_b1_test, average='macro')
print(f"Baseline 1 (Majority Class) - Train Macro F1: {b1_train_mf1:.4f}")
print(f"Baseline 1 (Majority Class) - Test Macro F1:  {b1_test_mf1:.4f}")


Baseline 1 (Majority Class) - Train Macro F1: 0.3333
Baseline 1 (Majority Class) - Test Macro F1:  0.3333


**Analysis (Majority Baseline)**
Predicting 0 ("No points") for every single driver is mathematically robust (because ~50% never score), but it utterly fails the Macro F1 metric metric (~0.33) because it earns a flat 0.0 F1 score for the positive class. This tells us what the absolute minimum acceptable score is when we predict blindly.

## 6. Model 2: Domain Heuristic Baseline (Grid Position <= 10)
**Justification:** A domain-specific heuristic. In F1, starting position often predicts finishing position due to track difficulty in overtaking. We predict "points" if the driver started 10th or better.


In [10]:
class GridHeuristicBaseline:
    def fit(self, X, y):
        pass # No training needed
    def predict(self, X):
        return (X['grid'] <= 10).astype(int)

baseline_2 = GridHeuristicBaseline()

y_pred_b2_train = baseline_2.predict(X_train)
y_pred_b2_test = baseline_2.predict(X_test)

b2_train_mf1 = f1_score(y_train, y_pred_b2_train, average='macro')
b2_test_mf1 = f1_score(y_test, y_pred_b2_test, average='macro')
print(f"Baseline 2 (Grid <= 10) - Train Macro F1: {b2_train_mf1:.4f}")
print(f"Baseline 2 (Grid <= 10) - Test Macro F1:  {b2_test_mf1:.4f}")


Baseline 2 (Grid <= 10) - Train Macro F1: 0.7448
Baseline 2 (Grid <= 10) - Test Macro F1:  0.7400


**Analysis (Domain Heuristic Baseline)**
This baseline performs incredibly well on the test set (`~0.74`). It proves what paddock experts know: track position heavily dictates finish position, as Formula 1 cars suffer aerodynamic penalties globally when following another car out of the top 10. Our ML algorithms must add actionable nuance beyond this rule to be worth deploying.

## 7. Model 3: Logistic Regression
**Justification:** A simple linear model combining grid position and constructor strength. Often a strong approach when classes are roughly balanced and relationships are linear.


In [11]:
model_lr = LogisticRegression(random_state=RANDOM_SEED, max_iter=1000)
model_lr.fit(X_train, y_train)

y_pred_lr_train = model_lr.predict(X_train)
y_pred_lr_test = model_lr.predict(X_test)

lr_train_mf1 = f1_score(y_train, y_pred_lr_train, average='macro')
lr_test_mf1 = f1_score(y_test, y_pred_lr_test, average='macro')
print(f"Logistic Regression - Train Macro F1: {lr_train_mf1:.4f}")
print(f"Logistic Regression - Test Macro F1:  {lr_test_mf1:.4f}")


Logistic Regression - Train Macro F1: 0.7398
Logistic Regression - Test Macro F1:  0.7599


**Analysis (Logistic Regression)**
This linear strategy yields noticeable improvements over our grid heuristic (`~0.76` Test F1 vs `~0.74`). By looking at both the grid position **and** the specific constructor power, the LR model accurately weights when a fast team starting 14th will inevitably push back up to score points (e.g. Red Bull or Mercedes).

## 8. Model 4: Random Forest
**Justification:** An ensemble non-linear model to capture interactions between specific constructors and their average grid performance.


In [12]:
model_rf = RandomForestClassifier(random_state=RANDOM_SEED, max_depth=5, n_estimators=100)
model_rf.fit(X_train, y_train)

y_pred_rf_train = model_rf.predict(X_train)
y_pred_rf_test = model_rf.predict(X_test)

rf_train_mf1 = f1_score(y_train, y_pred_rf_train, average='macro')
rf_test_mf1 = f1_score(y_test, y_pred_rf_test, average='macro')
print(f"Random Forest - Train Macro F1: {rf_train_mf1:.4f}")
print(f"Random Forest - Test Macro F1:  {rf_test_mf1:.4f}")


Random Forest - Train Macro F1: 0.8093
Random Forest - Test Macro F1:  0.7591


**Analysis (Random Forest)**
This is exactly the danger sign we look for in temporal holdouts. The tree-based algorithm achieves an outstanding `~0.81` F1 on the train set (2021-2022) but drops back to `~0.76` on the unseen 2023 test set. This drop (~5 F1 points) indicates overfitting: the Random Forest is memorizing the specific layout and hierarchy of the older seasons, rather than just learning the general rules of motor racing.

## 9. Final Comparison & Reasoning
**Justification:** We organize the metrics into a clear DataFrame to comply with C1 requirements (Train metric + Test metric, consistent evaluation). The reasoning is included via analysis of the train-test gaps and test performances.


In [13]:
results = pd.DataFrame([
    {"Model": "Majority Class", "Train Macro F1": b1_train_mf1, "Test Macro F1": b1_test_mf1, 
     "WHY (Mechanistic Reasoning)": "Merely outputs 0 (no points) blindly; test score reflects severe class imbalance penalty."},
    {"Model": "Grid Heuristic (<=10)", "Train Macro F1": b2_train_mf1, "Test Macro F1": b2_test_mf1, 
     "WHY (Mechanistic Reasoning)": "Extremely robust baseline. Since overtaking is hard, grid position heavily maps to points regardless of year. No overfitting (train ≈ test)."},
    {"Model": "Logistic Regression", "Train Macro F1": lr_train_mf1, "Test Macro F1": lr_test_mf1, 
     "WHY (Mechanistic Reasoning)": "Effectively weighted the grid starting position while assigning 'boosts' to historical top constructors like Red Bull/Mercedes, increasing stability."},
    {"Model": "Random Forest", "Train Macro F1": rf_train_mf1, "Test Macro F1": rf_test_mf1, 
     "WHY (Mechanistic Reasoning)": "Slight overfitting is visible (train F1 > test F1). Deep trees might rely on short-term 2022 patterns that fail to generalize fully into 2023 grid shifts."}
])

display(results)


,Model,Train Macro F1,Test Macro F1,WHY (Mechanistic Reasoning)
0,Majority Class,0.333333,0.333333,Merely outputs 0 (no points) blindly; test sco...
1,Grid Heuristic (<=10),0.744841,0.740000,Extremely robust baseline. Since overtaking is...
2,Logistic Regression,0.739766,0.759904,Effectively weighted the grid starting positio...
3,Random Forest,0.809314,0.759133,Slight overfitting is visible (train F1 > test...


In [17]:
# Export The Comparison Table automatically
# This fulfills the rubric requirement for a standalone comparison_table.md
try:
    with open('comparison_table.md', 'w', encoding='utf-8') as f:
        f.write("# Model Comparison Table\\n\\n")
        results.to_markdown(buf=f, index=False)
    print("Succesfully saved standalone comparison_table.md!")
except Exception as e:
    print("Make sure you run the results dataframe cell first.", e)

Succesfully saved standalone comparison_table.md!
